In [1]:
import os
import sys
from pathlib import Path

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "scripts").is_dir() else _cwd.parent
sys.path.insert(0, str(ROOT / "scripts"))

from notebook_setup import bootstrap

ROOT, DATA_DIR, PAPER_DIR = bootstrap()

In [2]:
import re
import requests

# 0050 成分股備援清單（每季調整時手動更新）
FALLBACK_TW50 = ['2330', '2454', '2308', '2317', '3711',
                 '2303', '2383', '2327', '2891', '3037', 
                 '2345', '2881', '2882', '2382', '2360', 
                 '3017', '1303', '2885', '2887', '2344', 
                 '2412', '2884', '2886', '2357', '2408', 
                 '2890', '6669', '2883', '3231', '3008', 
                 '4958', '2301', '2368', '2059', '7769', 
                 '3443', '1216', '2449', '2892', '3665', 
                 '2880', '3661', '3653', '5880', '2395', 
                 '2603', '4904', '8046', '3045', '6505']

YUANTA_RATIO_URL = "https://www.yuantaetfs.com/product/detail/0050/ratio"


def _split_nuxt_args(args_str: str) -> list[str]:
    """依逗號切分 Nuxt invoke 參數（尊重引號與括號嵌套）。"""
    parts: list[str] = []
    buf: list[str] = []
    depth = 0
    in_str = False
    esc = False
    quote = ""

    for ch in args_str:
        if in_str:
            buf.append(ch)
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue
        if ch in "\"'":
            in_str = True
            quote = ch
            buf.append(ch)
            continue
        if ch in "[{(":
            depth += 1
            buf.append(ch)
            continue
        if ch in "]})":
            depth -= 1
            buf.append(ch)
            continue
        if ch == "," and depth == 0:
            parts.append("".join(buf).strip())
            buf = []
            continue
        buf.append(ch)

    if buf:
        parts.append("".join(buf).strip())
    return parts


def _parse_nuxt_token(token: str):
    token = token.strip()
    if token == "null":
        return None
    if token == "true":
        return True
    if token == "false":
        return False
    if (token.startswith('"') and token.endswith('"')) or (
        token.startswith("'") and token.endswith("'")
    ):
        inner = token[1:-1]
        return bytes(inner, "utf-8").decode("unicode_escape")
    if re.fullmatch(r"-?\d+\.\d+", token):
        return float(token)
    if re.fullmatch(r"-?\d+", token):
        return int(token)
    return token


def _extract_invoke_args(nuxt_raw: str, invoke_idx: int) -> str:
    """括號配對取出 invoke 參數（忽略字串內的括號）。"""
    args_start = invoke_idx + 2
    depth = 1
    in_str = False
    esc = False
    quote = ""

    for i, ch in enumerate(nuxt_raw[args_start:], start=args_start):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue
        if ch in "\"'":
            in_str = True
            quote = ch
            continue
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            if depth == 0:
                return nuxt_raw[args_start:i]
    raise ValueError("找不到 invoke 參數結尾")


def _build_nuxt_param_map(nuxt_raw: str) -> dict:
    """解析 window.__NUXT__ 的參數對照表。"""
    invoke_idx = nuxt_raw.rfind("}(")
    if invoke_idx < 0:
        raise ValueError("找不到 __NUXT__ 參數區")

    fn_start = nuxt_raw.find("function(")
    fn_end = nuxt_raw.find("){")
    params = nuxt_raw[fn_start + 9: fn_end].split(",")

    args_str = _extract_invoke_args(nuxt_raw, invoke_idx)
    values = [_parse_nuxt_token(t) for t in _split_nuxt_args(args_str)]
    if len(values) != len(params):
        raise ValueError(
            f"Nuxt 參數數量不符：params={len(params)}, values={len(values)}"
        )
    return dict(zip(params, values))


def get_tw50_components():
    """從元大「基金權重-股票」抓取 0050 成分股商品代碼。

    頁面用 div 排版，不能用 pd.read_html。
    完整 50 檔資料在 SSR 的 window.__NUXT__.StockWeights 裡。
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Accept-Language": "zh-TW,zh;q=0.9",
    }

    try:
        resp = requests.get(YUANTA_RATIO_URL, headers=headers, timeout=20)
        resp.raise_for_status()
        html = resp.text

        marker = "window.__NUXT__="
        start = html.find(marker)
        if start < 0:
            raise ValueError("找不到 __NUXT__ 資料")

        nuxt_raw = html[start + len(marker): html.find("</script>", start)]
        param_map = _build_nuxt_param_map(nuxt_raw)

        sw_idx = nuxt_raw.find("StockWeights:[")
        if sw_idx < 0:
            raise ValueError("找不到 StockWeights")

        # 以括號配對找出陣列真正結束位置，避免固定視窗截斷漏抓最後幾檔
        arr_start = sw_idx + len("StockWeights:[")
        depth = 1
        arr_end = len(nuxt_raw)
        for i, ch in enumerate(nuxt_raw[arr_start:], start=arr_start):
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    arr_end = i
                    break

        code_vars = re.findall(
            r"code:([a-zA-Z_$][\w$]*)",
            nuxt_raw[sw_idx:arr_end],
        )
        stock_ids = [
            str(param_map[var])
            for var in code_vars
            if var in param_map and re.fullmatch(r"\d{4}", str(param_map[var]))
        ]

        if len(stock_ids) < 40:
            raise ValueError(f"解析到的股票數不足：{len(stock_ids)}")

        if len(stock_ids) != 50:
            missing = sorted(set(FALLBACK_TW50) - set(stock_ids))
            extra = sorted(set(stock_ids) - set(FALLBACK_TW50))
            print(f"⚠ 本次解析到 {len(stock_ids)} 檔，非預期的 50 檔")
            if missing:
                print(f"   相較備援清單，可能缺少：{missing}")
            if extra:
                print(f"   相較備援清單，多出（可能為成分股調整）：{extra}")

        print(f"已從元大持股比重頁取得 {len(stock_ids)} 檔成分股")
        return stock_ids
    except Exception as exc:
        print(f"元大網頁抓取失敗：{exc}")
        print(f"改用備援清單（共 {len(FALLBACK_TW50)} 檔）")
        return FALLBACK_TW50.copy()


tw50 = get_tw50_components()
print(tw50)

已從元大持股比重頁取得 50 檔成分股
['2330', '2454', '2308', '2317', '3711', '2303', '2383', '2327', '2891', '2345', '3037', '2881', '2382', '2882', '1303', '2887', '2885', '3017', '2360', '2886', '2890', '2412', '2884', '2344', '6669', '2883', '2357', '2059', '2408', '3231', '4958', '2301', '2368', '1216', '2892', '2880', '3008', '3443', '7769', '3665', '2449', '3661', '3653', '5880', '2395', '8046', '2603', '4904', '3045', '6505']


In [3]:
from FinMind.data import DataLoader
import pandas as pd
from datetime import datetime, timedelta
import time


api = DataLoader()
api.login_by_token(api_token=os.environ["FINMIND_TOKEN"])

# 今日
end_date = datetime.today().strftime("%Y-%m-%d")

# 10年前
start_date = (datetime.today() - timedelta(days=365*16)).strftime("%Y-%m-%d")



tw50 = get_tw50_components()

BATCH_SIZE = 10      # 每批撈幾檔
PAUSE_SECONDS = 3    # 每批之間停幾秒，避免 API 被限流
all_frames = []
for i in range(0, len(tw50), BATCH_SIZE):
    batch = tw50[i:i + BATCH_SIZE]
    print(f"正在撈第 {i // BATCH_SIZE + 1} 批: {batch}")
    batch_data = pd.concat([
        api.taiwan_stock_daily(
            stock_id=sid,
            start_date=start_date,
            end_date=end_date
        ).assign(stock_id=sid)
        for sid in batch
    ], ignore_index=True)
    all_frames.append(batch_data)
    # 最後一批不用等
    if i + BATCH_SIZE < len(tw50):
        print(f"等待 {PAUSE_SECONDS} 秒...")
        time.sleep(PAUSE_SECONDS)
stock_data = pd.concat(all_frames, ignore_index=True)
print(f"完成！共 {stock_data['stock_id'].nunique()} 檔，{len(stock_data)} 筆資料")


2026-07-08 18:07:43.134 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-07-08 18:07:43.213 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-07-08 18:07:44.458 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330


已從元大持股比重頁取得 50 檔成分股
正在撈第 1 批: ['2330', '2454', '2308', '2317', '3711', '2303', '2383', '2327', '2891', '2345']


2026-07-08 18:07:45.405 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2454
2026-07-08 18:07:45.935 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2308
2026-07-08 18:07:46.296 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-07-08 18:07:46.573 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 3711
2026-07-08 18:07:46.790 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2303
2026-07-08 18:07:47.077 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2383
2026-07-08 18:07:47.633 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2327
2026-07-08 18:07:48.055 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_i

等待 3 秒...


2026-07-08 18:07:51.962 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 3037


...

等待 3 秒...


2026-07-08 18:08:11.591 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2449


正在撈第 5 批: ['2449', '3661', '3653', '5880', '2395', '8046', '2603', '4904', '3045', '6505']


2026-07-08 18:08:12.267 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 3661
2026-07-08 18:08:12.523 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 3653
2026-07-08 18:08:12.888 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 5880
2026-07-08 18:08:13.103 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2395
2026-07-08 18:08:13.346 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 8046
2026-07-08 18:08:13.589 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2603
2026-07-08 18:08:13.817 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 4904
2026-07-08 18:08:14.042 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_i

完成！共 50 檔，187570 筆資料


In [4]:
try:
    DATA_DIR
except NameError:
    import sys
    from pathlib import Path

    _cwd = Path.cwd()
    ROOT = _cwd if (_cwd / "scripts").is_dir() else _cwd.parent
    sys.path.insert(0, str(ROOT / "scripts"))
    from notebook_setup import bootstrap

    ROOT, DATA_DIR, PAPER_DIR = bootstrap()

stock_data.to_csv(DATA_DIR / "TSMC_stock_data.csv", index=False)